In [ ]:
from pathlib import Path
import gc
import pandas as pd
from pycaret.classification import setup, compare_models, tune_model, blend_models, save_model

DATA_DIR = Path("../../data/analysis_ready")
OUTPUT_DIR = Path("../../outputs/specific_complication_models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SESSION_ID = 123
N_JOBS = 20
TUNE_ITERATIONS = 50
TARGET = "specific_complication"


In [ ]:
MODEL_ID = 'emer_med_2_dialysis_arf'
OUTCOME_NAME = 'Acute renal failure requiring dialysis'
development = pd.read_csv(DATA_DIR / f"{MODEL_ID}_development.csv")
validation = pd.read_csv(DATA_DIR / f"{MODEL_ID}_validation.csv")
required = {TARGET}
if not required.issubset(development.columns) or not required.issubset(validation.columns):
    raise ValueError(f"Both input files must contain {TARGET}")
feature_columns = [column for column in development.columns if column != TARGET]
if set(feature_columns) != set(column for column in validation.columns if column != TARGET):
    raise ValueError("Development and validation feature columns do not match")
validation = validation[feature_columns + [TARGET]]


In [ ]:
experiment = setup(
    data=development,
    target=TARGET,
    session_id=SESSION_ID,
    train_size=0.9999,
    preprocess=False,
    low_variance_threshold=0.1,
    remove_multicollinearity=True,
    multicollinearity_threshold=0.9,
    fold=5,
    fold_strategy="stratifiedkfold",
    fold_shuffle=True,
    verbose=False,
    n_jobs=N_JOBS,
)
candidate_models = compare_models(
    n_select=7,
    sort="AUC",
    exclude=["ridge", "lightgbm", "svm"],
)
tuned_models = [
    tune_model(
        model,
        optimize="AUC",
        n_iter=TUNE_ITERATIONS,
        choose_better=True,
    )
    for model in candidate_models
]
final_model = blend_models(
    estimator_list=tuned_models,
    optimize="AUC",
    method="auto",
)
model_directory = OUTPUT_DIR / MODEL_ID
model_directory.mkdir(parents=True, exist_ok=True)
save_model(final_model, str(model_directory / "final_blended_model"))
gc.collect()


In [ ]:
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score, roc_curve
from pycaret.classification import predict_model
import matplotlib.pyplot as plt

def evaluate_model(model, frame, dataset):
    predictions = predict_model(model, data=frame, raw_score=True, verbose=False)
    score_column = "prediction_score_1" if "prediction_score_1" in predictions.columns else "prediction_score"
    y_true = predictions[TARGET].astype(int).to_numpy()
    y_prob = predictions[score_column].astype(float).to_numpy()
    return {
        "Dataset": dataset,
        "AUROC": roc_auc_score(y_true, y_prob),
        "AUPRC": average_precision_score(y_true, y_prob),
        "Brier": brier_score_loss(y_true, y_prob),
    }, y_true, y_prob

development_result, development_y, development_probability = evaluate_model(final_model, development, "Training")
validation_result, validation_y, validation_probability = evaluate_model(final_model, validation, "Validation")
pd.DataFrame([development_result, validation_result]).to_csv(model_directory / "performance.csv", index=False)
pd.DataFrame({"y_true": development_y, "y_prob": development_probability}).to_csv(model_directory / "training_predictions.csv", index=False)
pd.DataFrame({"y_true": validation_y, "y_prob": validation_probability}).to_csv(model_directory / "validation_predictions.csv", index=False)

figure, axes = plt.subplots(1, 2, figsize=(11, 5))
for axis, title, y_true, y_prob in zip(
    axes,
    ("Training", "Validation"),
    (development_y, validation_y),
    (development_probability, validation_probability),
):
    false_positive_rate, true_positive_rate, _ = roc_curve(y_true, y_prob)
    axis.plot(false_positive_rate, true_positive_rate, label=f"AUROC = {roc_auc_score(y_true, y_prob):.2f}")
    axis.plot([0, 1], [0, 1], "--", color="0.6")
    axis.set(xlabel="1 - Specificity", ylabel="Sensitivity", title=title, xlim=(0, 1), ylim=(0, 1))
    axis.legend(frameon=False)
figure.suptitle(OUTCOME_NAME)
figure.tight_layout()
figure.savefig(model_directory / "roc_training_validation.png", dpi=300, bbox_inches="tight")
plt.close(figure)
